In [112]:
#%pip install numpy matplotlib pandas scipy

In [89]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import re

In [90]:
def read_data(file_path: str):
  data = pd.read_csv(file_path)
  return data

In [ ]:
def process(file_path: str, powerjoular: bool):
  data = read_data(file_path)
  if powerjoular:
    data = data[data["CPU Power"] > 0.000]
    data = data["CPU Power"]
  else:
    data = data[data["W"] > 0.000]
    data = data["W"]
  
  q25, q75 = data.quantile([0.25, 0.75])
  iqr = q75 - q25
  data = data[data.between(q25 - 1.5*iqr, q75 + 1.5*iqr)]
  filtered_data = data.sample(n=300)
  test_num = "".join(re.findall("test-\d", file_path))
  if test_num == '':
    test_num = "idle"
  return filtered_data, test_num

In [143]:
import os

powerjoular_alto_files = os.listdir("./experiment_results_alto/output_powerjoular_alto")
powerjoular_rundler_files = os.listdir("./experiment_results_rundler/output_powerjoular_rundler")
powerjoular_alto_process = []
powerjoular_rundler_process = []

for file in powerjoular_alto_files:
  if ".csv" in file:
    powerjoular_alto_process.append(file)

for file in powerjoular_rundler_files:
  if ".csv" in file:
    powerjoular_rundler_process.append(file)
    
print(powerjoular_alto_process)
print(powerjoular_rundler_process)
print(len(powerjoular_alto_process))

['powerjoular-raspi-alto-idle-160232.csv', 'powerjoular-test-6-raspi-r50-s25-t25-b12-156944.csv', 'powerjoular-test-3-raspi-r50-s100-t100-b12-155848.csv', 'powerjoular-test-8-raspi-r300-s100-t25-b2-159202.csv', 'powerjoular-test-2-raspi-r50-s100-t50-b12-155399.csv', 'powerjoular-test-5-raspi-r50-s50-t25-b12-156644.csv', 'powerjoular-test-7-raspi-r100-s100-t25-b6-157285.csv', 'powerjoular-test-4-raspi-r50-s75-t25-b12-156329.csv', 'powerjoular-test-1-raspi-r50-s100-t25-b12-155082.csv']
['powerjoular-test-3-raspi-rundler-r50-s100-t100-b12-174631.csv', 'powerjoular-raspi-rundler-idle-176719.csv', 'powerjoular-test-7-raspi-rundler-r100-s100-t25-b6-175955.csv', 'powerjoular-test-6-raspi-rundler-r50-s25-t25-b12-175587.csv', 'powerjoular-test-8-raspi-rundler-r300-s100-t25-b2-176314.csv', 'powerjoular-test-1-raspi-rundler-r50-s100-t25-b12-173700.csv', 'powerjoular-test-2-raspi-rundler-r50-s100-t50-b12-174305.csv', 'powerjoular-test-4-raspi-rundler-r50-s75-t25-b12-174958.csv', 'powerjoular-test-

In [144]:
def process_files(files: list[str], path: str, powerjoular: bool = True):
  data_dict = {}
  for f in files:
    df, num_cpu = process(f"{path}{f}", powerjoular)
    data_dict[num_cpu] = df
  return data_dict

In [145]:
powerjoular_alto_data = process_files(powerjoular_alto_process, "./experiment_results_alto/output_powerjoular_alto/")
powerjoular_rundler_data = process_files(powerjoular_rundler_process, "./experiment_results_rundler/output_powerjoular_rundler/")

In [146]:
def print_summary_powerspy(title, data_dict):
  print(title)
  for k, v in data_dict.items():
    mean = v["W"].mean()
    print(f"{k}, mean {mean}")

In [147]:
def print_summary_powerjoular(title, data_dict):
  print(title)
  for k, v in data_dict.items():
    mean = v.mean()
    print(f"{k}, mean {mean}")

In [148]:
print_summary_powerjoular("Powerjoular alto summary", powerjoular_alto_data)
print_summary_powerjoular("Powerjoular rundler summary", powerjoular_rundler_data)

Powerjoular alto summary
idle, mean 2.5123886955944053
test-6, mean 3.2104523326845937
test-3, mean 2.99812676971245
test-8, mean 3.3124129665615816
test-2, mean 3.136879303977704
test-5, mean 3.265424428884259
test-7, mean 3.3184115036612534
test-4, mean 3.267726733540816
test-1, mean 3.3242712479347087
Powerjoular rundler summary
test-3, mean 3.444603302516631
idle, mean 2.776420459899559
test-7, mean 3.4453125092358867
test-6, mean 3.4878552076091607
test-8, mean 3.410012269309027
test-1, mean 3.4696169895029243
test-2, mean 3.4432517289807802
test-4, mean 3.4478135056024013
test-5, mean 3.375716295991128


In [149]:
for k, v in powerjoular_alto_data.items():
  print(v.head())

374    1.674925
53     1.674859
323    1.674925
260    1.674925
80     1.674859
Name: CPU Power, dtype: float64
312    3.081431
328    3.274061
725    3.376945
281    2.844745
506    3.478870
Name: CPU Power, dtype: float64
4332    2.712776
5085    2.384773
4263    2.826212
5077    3.116055
5364    3.067200
Name: CPU Power, dtype: float64
3258    3.279306
5127    3.148810
665     3.351186
509     3.226177
2323    3.455139
Name: CPU Power, dtype: float64
1167    2.915283
1876    2.621985
1719    3.284166
534     2.926356
1066    3.061640
Name: CPU Power, dtype: float64
168    3.943148
364    3.290329
309    2.922776
504    2.819064
385    3.837780
Name: CPU Power, dtype: float64
2060    3.358812
976     3.594751
971     3.387236
1008    3.476838
2150    3.376875
Name: CPU Power, dtype: float64
613     3.386869
1080    3.438237
295     3.376875
906     3.376875
793     3.519165
Name: CPU Power, dtype: float64
811    3.376875
805    3.592678
724    3.536181
885    3.528148
919    3.244686